In [ ]:
import os
import glob
import numpy as np
import xarray as xr

import matplotlib
from matplotlib import cm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1 import make_axes_locatable
%matplotlib inline

from joblib import Parallel, delayed

# --- gridded NetCDF + per-basin river profiles (PyGMT/ArcGIS + matplotlib) ---
from gospl.analyse.gridexport import (
    grid_export, to_netcdf, basin_rivers, plot_long_profile, plot_basin_map)

# --- stratigraphic sections / wells / Wheeler (matplotlib, inline) ---
from gospl.analyse.stratasection import (
    load_strata, cross_section, horizontal_slice, synthetic_well, wheeler,
    well_panel)

## Running the simulation

First activate the conda environment:

```bash
conda activate gospl
```

To run the simulation in a terminal (`X` = number of MPI processes, e.g. 5):

```bash
mpirun -np X gospl -i input.yml
```

# Analysing the outputs

All the post-processing below uses goSPL's built-in **`gospl.analyse`** toolkit
(imported in the first cell). Two complementary modules are used:

- **`gospl.analyse.gridexport`** — reassembles the unstructured mesh, rasterises
  every surface field of an output step onto a **regular grid**, runs D8
  hydrology (drainage area, basins, &chi;) and writes a CF-NetCDF for
  PyGMT/ArcGIS. It also exposes per-basin **river long-profile** helpers.
- **`gospl.analyse.stratasection`** — reads the recorded stratigraphy and draws
  **cross-sections, synthetic wells and Wheeler (chronostratigraphic)
  diagrams** (coloured by facies, lithology, provenance, &hellip;).

Every function used below has a **terminal equivalent** so the same products can
be generated outside Jupyter. These console commands &mdash; `gospl-grid`,
`gospl-section`, `gospl-strata-volume` &mdash; are installed with goSPL and are
shown in each section.

Each gridded NetCDF (one file per output step) holds, when available (every
variable carries its `units` and a `long_name` definition):

+ surface elevation `elev` (m) and the step's `sea_level` (m)
+ cumulative erosion/deposition `erodep` (m) and its rate `EDrate` (m/yr)
+ water / sediment fluxes `FA`, `fillFA`, `waterFill`, `sedLoad`
+ hydrology: `drainage_area`, `basin` id, `chi`, `flowdist`, and the
  priority-flood-`filled` elevation

In [ ]:
# Define output folder name for the simulation
out_path = 'results/'

if not os.path.exists(out_path):
    os.makedirs(out_path)

### Rasterising the outputs to a regular grid &mdash; `grid_export` / `to_netcdf`

`grid_export` reassembles the global mesh, interpolates a step's fields onto a
regular grid, runs the D8 hydrology and returns a dict of 2-D arrays;
`to_netcdf` writes that to a CF-NetCDF (each variable annotated with its `units`
and `long_name`). `getOutputs` below simply loops over the steps and writes one
`results/surface<step>.nc` per step.

**`grid_export(h5dir, mesh, step=None, ...)` &mdash; main options**

| Argument | Default | Meaning |
|---|---|---|
| `h5dir` | &ndash; | the run's `h5` output directory |
| `mesh` | &ndash; | global mesh `.npz` (vertices `v`, cells `c`) |
| `step` | last | output step to rasterise |
| `spacing` | median edge | grid resolution `dx[,dy]` (mesh units) |
| `fields` | all | subset of surface fields to include |
| `mn` | `0.5` | &chi; concavity `m/n` |
| `a0` | `1.0` | &chi; reference drainage area |
| `base_level` | run sea level | elevation defining the coast / outlets (catchment + &chi; datum) |
| `latlim` | `89` | (global meshes) crop the polar caps |

Global (spherical) meshes are auto-detected and gridded in lon/lat. The resolved
sea level is stored in each file (global attribute **and** a `sea_level`
variable), so the grid is self-describing.

**Terminal equivalent** (one step &rarr; one NetCDF):

```bash
gospl-grid --h5dir sim_gw_geochem/h5 --mesh data/gospl_mesh.npz:v:c \
    --step 10 --spacing 250 --out results/surface10.nc
```

For the whole time series, loop in the shell:

```bash
for s in $(seq 0 11); do
  gospl-grid --h5dir sim_gw_geochem/h5 --mesh data/gospl_mesh.npz:v:c \
      --step $s --spacing 250 --out results/surface$s.nc
done
```

In [ ]:
h5dir = "sim_gw_geochem/h5"
mesh = "data/gospl_mesh.npz"
reso = 250

out_name = "surface"

def getOutputs(steps):

    # clear any stale .nc files first
    for f in glob.glob(os.path.join(out_path, f"{out_name}*.nc")):
        try:
            os.remove(f)
        except PermissionError:
            print(f"Still locked, close it first: {f}")
            return
        
    for stp in steps:
        g = grid_export(h5dir, mesh, stp, spacing=reso)
        fname = os.path.join(out_path, f"{out_name}{stp}.nc")
        to_netcdf(g, fname)                       

    return

def getOutputsParallel(steps, n_workers=8):
    for f in glob.glob(os.path.join(out_path, f"{out_name}*.nc")):
        try:
            os.remove(f)
        except PermissionError:
            print(f"Still locked, close it first: {f}")
            return

    def process_step(stp):
        g = grid_export(h5dir, mesh, stp, spacing=reso)
        fname = os.path.join(out_path, f"{out_name}{stp}.nc")
        to_netcdf(g, fname)

    Parallel(n_jobs=n_workers)(delayed(process_step)(stp) for stp in steps)

steps = np.arange(11)
getOutputsParallel(steps, n_workers=8)
# getOutputs(steps)

### Surface elevation through time

The four panels show the remapped `elevation` field at steps 5, 10, 15 and 25, with the black contour marking the $0$ m shoreline. Watch how the coastline migrates as the prescribed sea level and sediment supply reshape the margin: a seaward-stepping shoreline indicates progradation, a landward-stepping one indicates transgression.

In [ ]:
ncfiles = [os.path.join(out_path, f"{out_name}{stp}.nc") for stp in steps]
ds = {stp: xr.open_dataset(f) for stp, f in zip(steps, ncfiles)}
ds[5]

In [ ]:
stps = [2,5,7,10]
fig, axs = plt.subplots(2,2, figsize=(8,8), sharex=True, sharey=True)
for ax, stp in zip(axs.flat, stps):
    im = ds[stp].elev.plot(ax=ax, add_labels=False, add_colorbar=False, cmap='Spectral_r')
    ds[stp].elev.plot.contour(ax=ax, levels=[ds[stp].sea_level.values], colors=['k'])
for ax, stp in zip(axs.flat, stps):
    ax.set_title(f'step = {stp}', fontsize=10, fontweight="bold")
cbar_ax = fig.add_axes([0.2, -0.02, 0.6, 0.02]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Elevation (m)')
plt.show()

fig, axs = plt.subplots(2,2, figsize=(8,8), sharex=True, sharey=True)
for ax, stp in zip(axs.flat, stps):
    im = ds[stp].erodep.plot(ax=ax, add_labels=False, add_colorbar=False, cmap='bwr', vmin=-100, vmax=100)
    ds[stp].elev.plot.contour(ax=ax, levels=[ds[stp].sea_level.values], colors=['k'])
for ax, stp in zip(axs.flat, stps):
    ax.set_title(f'step = {stp}', fontsize=10, fontweight="bold")
cbar_ax = fig.add_axes([0.2, -0.02, 0.6, 0.02]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Erosion / deposition (m)')
plt.show()

fig, axs = plt.subplots(2,2, figsize=(8,8), sharex=True, sharey=True)
for ax, stp in zip(axs.flat, stps):
    im = ds[stp].soilH.plot(ax=ax, add_labels=False, add_colorbar=False, cmap='Oranges', vmin=0, vmax=3)
    ds[stp].elev.plot.contour(ax=ax, levels=[ds[stp].sea_level.values], colors=['k'])
for ax, stp in zip(axs.flat, stps):
    ax.set_title(f'step = {stp}', fontsize=10, fontweight="bold")
cbar_ax = fig.add_axes([0.2, -0.02, 0.6, 0.02]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Soil thickness (m)')
plt.show()

## Drainage basins and river long profiles

The gridded files already carry the `basin` ids and `chi`. To examine the
**channel network of a single basin**, `basin_rivers` traces the main stem (up
the largest-area donor at each step) and its tributaries; `plot_basin_map` maps
them with the **sea-level coastline**, and `plot_long_profile` draws the
longitudinal profile (distance in km).

**What you can do:** pick a basin (by id, or by clicking a point as below), set
the channel-defining drainage-area threshold, map the network over any gridded
field, and plot the long profile against distance or **&chi;** &mdash; either the
raw elevation (keeps real lakes as dips) or the hydrologically-`filled` one
(monotonic).

| Function | Key options | Meaning |
|---|---|---|
| `basin_rivers(result, ...)` | `basin_id`, `area_threshold` | basin to extract (default: largest); min drainage area (m&sup2;) for a channel (default: 95th pct) |
| `plot_basin_map(result, rivers, ...)` | `background`, `sea_level`, `figsize` | base field (default `elev`); coastline datum (default run sea level); figure size |
| `plot_long_profile(rivers, ...)` | `xaxis`, `which`, `figsize` | `dist` or `chi`; `elev` (raw) or `filled` (monotonic); figure size |

These per-basin plots are a **notebook API** (no dedicated console command); they
read the same grid `gospl-grid` produces. Below we first locate a basin id by
picking a point, then extract and plot its rivers.

In [ ]:
step = 10
fig, ax = plt.subplots(figsize=(6, 5))

elev = ds[step].elev
elev.plot(ax=ax, cmap="gray", alpha=0.2, add_colorbar=False)

basin = ds[step].basin
basin.plot(ax=ax, cmap="jet") #,vmax=00)
levels = np.unique(ds[step].basin.values)
cs = ax.contour(basin.x, basin.y, basin.values, levels=levels, colors="k", linewidths=0.1)
cs = ax.contour(basin.x, basin.y, elev.values, levels=[ds[step].sea_level], colors="k", linestyles='-', linewidths=1)

plt.tight_layout()
plt.show()

In [ ]:
xbasin = 4.2e5
ybasin = 4.6e6
basin_id = int(ds[step].sel(x=xbasin, y=ybasin, method='nearest').basin.values)
print(f"Corresponding basin ID: {basin_id}")

In [ ]:
g = grid_export(h5dir, mesh, step, spacing=reso)
riv = basin_rivers(g, basin_id=basin_id, area_threshold=5e6)
plot_basin_map(g, riv)
plot_long_profile(riv, which="elev")